<a href="https://colab.research.google.com/github/SheffieldML/notebook/blob/main/GPy/f1_2011_qualifying_cutoff.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Skew Likelihood for F1 qualifying cut-off time prediction with a driver/team GP

In 2011 we a the University of Sheffield we did some work with an F1 team predicting cut off times using Gaussian processes. This notebook recreates that work as a demonstration of skewed likelihoods in GPy. It uses a Gaussian process whose covariance is
built from practice pace, **team**, and **driver** consistency terms.


## Origin

The original code predicted qualifying cut-offs
from free-practice times:
$$
y_i = \log q_i - m\,\log p_i
$$
with inputs
$$
X_i = [\mathrm{driver},\ \mathrm{team},\ p_i,\ \log p_i,\ \mathrm{noise}_i].
$$
The compound covariance is

1. **Linear** on $\log p$ — calibration slope $m$ uncertainty
2. **RBF** on $\log p$ — smooth nonlinearity in pace
3. **Bias** — track-wide consistency
4. **IndexARD** on team — one variance per constructor
5. **IndexARD** on driver — fixed drivers pinned to their observed pace
6. **Diag** on a noise column — heteroscedastic driver consistency for the rest

This notebook implements that structure in GPy (`IndexARD` / `DiagFromInput` are
small local kernels matching GPmat `indexard` / `diag`), loads Silverstone
2011 via `pods.datasets.f1_2011_practice_qualifying`, and reports the
predicted **P17 cut-off** distribution (Q1 → Q2 in a 24-car field).


## Setup

Needs GPy (`gpy[plotting]>=1.15`) and pods with CIP-0005
(`f1_2011_practice_qualifying`). The next cell installs both if missing
(GPy from PyPI; pods from the ods CIP-0005 branch until that loader is on
PyPI). Tutorial CI uses the same sources via `installs.yml`.


In [ ]:
import importlib
import json
import shutil
import subprocess
import sys
from urllib.request import urlopen

# Shared defaults: lamd CIP-001C catalog. Temporary pods pin matches installs.yml.
_CATALOG_JSON = (
    "https://raw.githubusercontent.com/lawrennd/lamd/main/lamd/installs/packages.json"
)
_OVERRIDES = {
    "GPy": "gpy[plotting]>=1.15.0",
    "pods": "git+https://github.com/lawrennd/ods.git@cip0005-f1-practice-qualifying",
}


def _pip_install(spec):
    if shutil.which("uv"):
        cmd = ["uv", "pip", "install", "--python", sys.executable, "-q", spec]
    else:
        cmd = [sys.executable, "-m", "pip", "install", "-q", spec]
    subprocess.check_call(cmd)


def _spec(name):
    if name in _OVERRIDES:
        return _OVERRIDES[name]
    try:
        with urlopen(_CATALOG_JSON, timeout=10) as response:
            packages = json.loads(response.read().decode("utf-8")).get("packages", {})
        return packages.get(name, name)
    except Exception:
        return name


def _ensure(import_name, install_as=None):
    try:
        return importlib.import_module(import_name)
    except ImportError:
        _pip_install(_spec(install_as or import_name))
        return importlib.import_module(import_name)


np = _ensure("numpy")
_ensure("matplotlib")
from matplotlib import pyplot as plt

GPy = _ensure("GPy")
pods = _ensure("pods")

# Old PyPI pods lacks the 2011 F1 loader — force the CIP-0005 build.
if not hasattr(pods.datasets, "f1_2011_practice_qualifying"):
    _pip_install(_OVERRIDES["pods"])
    for key in list(sys.modules):
        if key == "pods" or key.startswith("pods."):
            del sys.modules[key]
    pods = importlib.import_module("pods")
    if not hasattr(pods.datasets, "f1_2011_practice_qualifying"):
        raise ImportError(
            "pods.datasets.f1_2011_practice_qualifying still missing after "
            f"installing {_OVERRIDES['pods']!r}"
        )


## Local kernels

GPy has no stock IndexARD. These match the MATLAB semantics used in
`createGpModel`: same discrete label → shared scale; diagonal-from-input for
per-driver noise.


In [ ]:
class IndexARD(GPy.kern.Kern):
    """Covariance: K_ij = scale[x_i] if round(x_i)==round(x_j) else 0."""

    def __init__(self, scales_by_index, active_dims=None, name="indexard"):
        super().__init__(input_dim=1, active_dims=active_dims, name=name)
        self.scales_by_index = {int(k): float(v) for k, v in scales_by_index.items()}

    def K(self, X, X2=None):
        x = np.round(np.asarray(X)).astype(int).ravel()
        x2 = x if X2 is None else np.round(np.asarray(X2)).astype(int).ravel()
        sx = np.array([self.scales_by_index.get(int(i), 0.0) for i in x], dtype=float)
        return (x[:, None] == x2[None, :]) * sx[:, None]

    def Kdiag(self, X):
        x = np.round(np.asarray(X)).astype(int).ravel()
        return np.array([self.scales_by_index.get(int(i), 0.0) for i in x], dtype=float)

    def update_gradients_full(self, dL_dK, X, X2=None):
        pass

    def update_gradients_diag(self, dL_dKdiag, X):
        pass


class DiagFromInput(GPy.kern.Kern):
    """Covariance: K_ii = variance * exp(x_i), off-diagonal 0 (GPmat diag)."""

    def __init__(self, variance=1.0, active_dims=None, name="diag"):
        super().__init__(input_dim=1, active_dims=active_dims, name=name)
        self.variance = float(variance)

    def K(self, X, X2=None):
        if X2 is not None:
            return np.zeros((X.shape[0], X2.shape[0]))
        v = self.variance * np.exp(np.asarray(X, dtype=float).ravel())
        v = np.where(np.isfinite(v), v, 0.0)
        return np.diag(v)

    def Kdiag(self, X):
        v = self.variance * np.exp(np.asarray(X, dtype=float).ravel())
        return np.where(np.isfinite(v), v, 0.0)

    def update_gradients_full(self, dL_dK, X, X2=None):
        pass

    def update_gradients_diag(self, dL_dKdiag, X):
        pass


## Load Silverstone 2011 (FP3 → Q1)

Default demo race in `regressGpPractice`. Cut-off rank **17** is the Q1→Q2
bubble in a 24-car field.


In [ ]:
GRAND_PRIX = "GreatBritain"
PRACTICE = "FP3"
Q_COL = 0  # Y columns: Q1, Q2, Q3
CUTOFF_RANK = 17

if not hasattr(pods.datasets, "f1_2011_practice_qualifying"):
    raise ImportError(
        "pods.datasets.f1_2011_practice_qualifying is missing. "
        "Install pods from ods CIP-0005, e.g. "
        "`python scripts/install_from_yaml.py --profile gpy-tutorials`."
    )

data = pods.datasets.f1_2011_practice_qualifying(
    grand_prix=GRAND_PRIX, practice=PRACTICE
)

p = np.asarray(data["X"], dtype=float).ravel()
Y = np.asarray(data["Y"], dtype=float)
q = Y[:, Q_COL]
driver_num = np.asarray(data["driver_number"], dtype=int).ravel()
team_idx = np.asarray(data["team_index"], dtype=int).ravel()  # 1..12 pit order
abbr = np.asarray(data["abbreviation"])

ok = np.isfinite(p) & np.isfinite(q)
p, q, driver_num, team_idx, abbr = p[ok], q[ok], driver_num[ok], team_idx[ok], abbr[ok]
print(f"{GRAND_PRIX} {PRACTICE}→Q1: {len(p)} drivers with both times")


## Consistency defaults

Percent-style standard deviations are converted to geometric variances with
$\log(1 + (\sigma/100)^2)$, matching MATLAB.


In [ ]:
def pct_var(std_pct):
    std_pct = np.asarray(std_pct, dtype=float)
    return np.log(1.0 + (std_pct * std_pct) / 10000.0)

# Defaults from regressGpPractice.m for Q1/Q2/Q3
m_mean = 0.8
m_std = 2.0
non_lin_std = 0.5
non_lin_width = float(np.std(np.log(p)))
track_std = 5.0
team_std = np.full(12, 0.5)
driver_std = np.ones(len(p))

# Alonso (5) and Massa (6) fixed to observed practice in the 2011 demos
fix_driver = np.isin(driver_num, [5, 6])

driver_var = pct_var(driver_std)  # used for fixed-driver IndexARD scales
Xnoise = np.full(len(p), -np.inf)
Xnoise[~fix_driver] = np.log(np.maximum(driver_var[~fix_driver], 1e-300))

log_p = np.log(p)
# createModelInput columns: [driverNum, teamNum, pLapTimes, logInput, Xnoise]
X = np.column_stack([driver_num, team_idx, p, log_p, Xnoise])
y = (np.log(q) - m_mean * log_p)[:, None]


## Build GP covariance and fit

Hyperparameters are set from the consistency structure; optimisation was typically left commented out for the live tool.


In [ ]:
team_scales = {t: float(pct_var(team_std[t - 1])) for t in range(1, 13)}
driver_scales = {
    int(d): (float(driver_var[i]) if fix_driver[i] else 0.0)
    for i, d in enumerate(driver_num)
}

kern = (
    GPy.kern.Linear(1, variances=m_std ** 2, active_dims=[3], name="lin")
    + GPy.kern.RBF(
        1,
        variance=float(pct_var(non_lin_std)),
        lengthscale=non_lin_width,
        active_dims=[3],
        name="rbf",
    )
    + GPy.kern.Bias(1, variance=float(pct_var(track_std)), active_dims=[3], name="track")
    + IndexARD(team_scales, active_dims=[1], name="team")
    + IndexARD(driver_scales, active_dims=[0], name="driver_fixed")
    + DiagFromInput(1.0, active_dims=[4], name="driver_noise")
)

model = GPy.models.GPRegression(X, y, kernel=kern, noise_var=1e-8)
model.Gaussian_noise.fix()
# Optional: model.optimize(messages=True)  # MATLAB usually skipped this live
print(model)


## Predicted vs actual Q1 times

Undo the log-link: $\hat q = \exp(\hat y + m\log p)$.


In [ ]:
y_mean, y_var = model.predict(X)
log_q_hat = y_mean.ravel() + m_mean * log_p
q_hat = np.exp(log_q_hat)
q_std = q_hat * np.sqrt(np.maximum(y_var.ravel(), 0.0))  # delta method

fig, ax = plt.subplots(figsize=(6, 6))
lo = min(q.min(), q_hat.min()) - 0.2
hi = max(q.max(), q_hat.max()) + 0.2
ax.plot([lo, hi], [lo, hi], "k--", lw=1, alpha=0.5)
ax.errorbar(q_hat, q, xerr=2 * q_std, fmt="o", ms=5, alpha=0.85)
for a, xh, ya in zip(abbr, q_hat, q):
    ax.annotate(a, (xh, ya), textcoords="offset points", xytext=(4, 2), fontsize=7)
ax.set_xlabel("Predicted Q1 (s)")
ax.set_ylabel("Actual Q1 (s)")
ax.set_title(f"{GRAND_PRIX} 2011: GP predictions from {PRACTICE}")
ax.set_aspect("equal", adjustable="box")
ax.set_xlim(lo, hi)
ax.set_ylim(lo, hi)
plt.show()

order = np.argsort(q_hat)
print("Predicted order (fast→slow):", " ".join(abbr[order]))


## Cut-off distribution (rank 17)

Draw posterior samples of $y$, map to lap times, sort each draw, and read
off the 17th-fastest time — the Q1 cut-off into Q2 (`visualizeGrid.m`).


In [ ]:
rng = np.random.default_rng(1)
n_samp = 2000
mu, cov = model.predict(X, full_cov=True)
cov = cov + 1e-8 * np.eye(len(p))
y_samp = rng.multivariate_normal(mu.ravel(), cov, size=n_samp).T  # (n, S)
q_samp = np.exp(y_samp + m_mean * log_p[:, None])

sorted_samp = np.sort(q_samp, axis=0)
cutoff_samp = sorted_samp[CUTOFF_RANK - 1, :]
cutoff_mean = float(np.mean(cutoff_samp))
cutoff_std = float(np.std(cutoff_samp))

danger = np.mean(q_samp > cutoff_samp[None, :], axis=1)

print(f"Mean cut-off (P{CUTOFF_RANK}): {cutoff_mean:.3f}s")
print(f"STD  cut-off (P{CUTOFF_RANK}): {cutoff_std:.4f}s")
print()
rank_by_mean = np.argsort(q_samp.mean(axis=1))
print(f"{'Rk':>3} {'Drv':>4} {'E[Q1]':>8} {'Danger17':>9}")
for r, i in enumerate(rank_by_mean, start=1):
    print(f"{r:3d} {abbr[i]:>4} {q_samp.mean(axis=1)[i]:8.3f} {100 * danger[i]:8.1f}%")

fig, ax = plt.subplots(figsize=(7, 3.5))
ax.hist(cutoff_samp, bins=40, density=True, color="C0", alpha=0.85)
ax.axvline(cutoff_mean, color="k", lw=1.5, label=f"mean {cutoff_mean:.3f}s")
ax.set_xlabel(f"Sampled P{CUTOFF_RANK} cut-off time (s)")
ax.set_ylabel("Density")
ax.set_title(f"{GRAND_PRIX} 2011 Q1 cut-off from {PRACTICE} GP")
ax.legend()
plt.show()


## Partial observations (cut-off tool spirit)

`sampleGrids.m` conditions on drivers who have already set a flying lap.
Here we pretend the top 10 Q1 times are known and re-predict the cut-off.


In [ ]:
known = np.argsort(q)[:10]
X_k, y_k = X[known], y[known]

model_partial = GPy.models.GPRegression(X_k, y_k, kernel=kern.copy(), noise_var=1e-8)
model_partial.Gaussian_noise.fix()

mu_all, cov_all = model_partial.predict(X, full_cov=True)
cov_all = cov_all + 1e-8 * np.eye(len(p))
y_s = rng.multivariate_normal(mu_all.ravel(), cov_all, size=n_samp).T
q_s = np.exp(y_s + m_mean * log_p[:, None])
for i in known:
    q_s[i, :] = np.minimum(q_s[i, :], q[i])

cut_s = np.sort(q_s, axis=0)[CUTOFF_RANK - 1]
print(
    f"With top-10 Q1 times observed: "
    f"cut-off mean {cut_s.mean():.3f}s  std {cut_s.std():.4f}s "
    f"(full-data cut-off was {cutoff_mean:.3f}s)"
)


## Notes

- Team indices come from the curated pack (`team_index`, pit-lane order 1–12),
  matching `num2teamNum` in the MATLAB demos.
- Fixed drivers 5 and 6 follow the 2011 demo defaults; change `fix_driver` to
  experiment.
- A fuller port of the live GUI (`cutoffPredictor`) would stream sector
  updates; this notebook is the static GP + cut-off sampling core.
- Data: `pods` /
  [datasets_mirror `f1_2011_practice_qualifying`](https://github.com/lawrennd/datasets_mirror/tree/main/f1_2011_practice_qualifying)
  ([ods CIP-0005](https://github.com/lawrennd/ods)).
